In [ ]:
# Required Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, make_scorer

import warnings
warnings.filterwarnings('ignore')

## 1. Prepare Data

In [ ]:
# Load and preprocess Titanic
df = pd.read_csv("../data/titanic.csv")

# Simple preprocessing
X = df.drop(columns=['Survived'])
y = df['Survived']

# Identify columns
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# Handle missing values
X[numeric_cols] = SimpleImputer(strategy='median').fit_transform(X[numeric_cols])
X[categorical_cols] = SimpleImputer(strategy='most_frequent').fit_transform(X[categorical_cols])

# Encode categoricals
for col in categorical_cols:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# Scale
X[numeric_cols] = StandardScaler().fit_transform(X[numeric_cols])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

## 2. Baseline Model

In [ ]:
# Train baseline Random Forest
baseline = RandomForestClassifier(random_state=42)
baseline.fit(X_train, y_train)

y_pred = baseline.predict(X_test)
baseline_acc = accuracy_score(y_test, y_pred)
baseline_f1 = f1_score(y_test, y_pred)

print("📊 Baseline Random Forest (default params):")
print(f"   Accuracy: {baseline_acc:.4f}")
print(f"   F1 Score: {baseline_f1:.4f}")

## 3. Grid Search

In [ ]:
# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

print("🔍 Grid Search Parameter Space:")
total_combinations = 1
for param, values in param_grid.items():
    print(f"   {param}: {values}")
    total_combinations *= len(values)
print(f"\n   Total combinations: {total_combinations}")

In [ ]:
# Run Grid Search
print("\n🏃 Running Grid Search (this may take a minute)...")

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\n✅ Grid Search Complete!")
print(f"\n🏆 Best Parameters: {grid_search.best_params_}")
print(f"   Best CV Score: {grid_search.best_score_:.4f}")

In [ ]:
# Evaluate on test set
y_pred_grid = grid_search.predict(X_test)
grid_acc = accuracy_score(y_test, y_pred_grid)
grid_f1 = f1_score(y_test, y_pred_grid)

print("📊 Grid Search Results on Test Set:")
print(f"   Accuracy: {grid_acc:.4f} (Baseline: {baseline_acc:.4f})")
print(f"   F1 Score: {grid_f1:.4f} (Baseline: {baseline_f1:.4f})")
print(f"\n   Improvement: {(grid_f1 - baseline_f1) * 100:.2f}% F1")

## 4. Random Search

In [ ]:
from scipy.stats import randint, uniform

# Define parameter distributions
param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': [3, 5, 10, 15, 20, None],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None]
}

print("🎲 Random Search Parameter Distributions:")
for param, values in param_dist.items():
    print(f"   {param}: {values}")

In [ ]:
# Run Random Search
print("\n🏃 Running Random Search (50 iterations)...")

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist,
    n_iter=50,  # Number of random combinations to try
    cv=5,
    scoring='f1',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train, y_train)

print("\n✅ Random Search Complete!")
print(f"\n🏆 Best Parameters: {random_search.best_params_}")
print(f"   Best CV Score: {random_search.best_score_:.4f}")

In [ ]:
# Evaluate on test set
y_pred_random = random_search.predict(X_test)
random_acc = accuracy_score(y_test, y_pred_random)
random_f1 = f1_score(y_test, y_pred_random)

print("📊 Random Search Results on Test Set:")
print(f"   Accuracy: {random_acc:.4f}")
print(f"   F1 Score: {random_f1:.4f}")

## 5. Optuna (Bayesian Optimization)

In [ ]:
# Install optuna if needed
try:
    import optuna
    print(f"✅ Optuna version: {optuna.__version__}")
except ImportError:
    print("Installing optuna...")
    !pip install optuna -q
    import optuna
    print(f"✅ Optuna installed: {optuna.__version__}")

In [ ]:
import optuna
from optuna.samplers import TPESampler

# Suppress optuna logs
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    """
    Optuna objective function for hyperparameter optimization.
    """
    # Define hyperparameter search space
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'random_state': 42
    }
    
    # Create and evaluate model
    model = RandomForestClassifier(**params)
    
    # Cross-validation score
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
    
    return scores.mean()

print("🧪 Optuna Hyperparameter Search Space:")
print("   n_estimators: [50, 300]")
print("   max_depth: [3, 20]")
print("   min_samples_split: [2, 20]")
print("   min_samples_leaf: [1, 10]")
print("   max_features: ['sqrt', 'log2', None]")

In [ ]:
# Create and run study
print("\n🏃 Running Optuna Study (50 trials)...")

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42)
)

study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n✅ Optuna Study Complete!")
print(f"\n🏆 Best Parameters: {study.best_params}")
print(f"   Best CV Score: {study.best_value:.4f}")

In [ ]:
# Train model with best parameters
best_params = study.best_params
best_params['random_state'] = 42

optuna_model = RandomForestClassifier(**best_params)
optuna_model.fit(X_train, y_train)

# Evaluate
y_pred_optuna = optuna_model.predict(X_test)
optuna_acc = accuracy_score(y_test, y_pred_optuna)
optuna_f1 = f1_score(y_test, y_pred_optuna)

print("📊 Optuna Results on Test Set:")
print(f"   Accuracy: {optuna_acc:.4f}")
print(f"   F1 Score: {optuna_f1:.4f}")

## 6. Compare All Methods

In [ ]:
# Comparison table
comparison = pd.DataFrame({
    'Method': ['Baseline', 'Grid Search', 'Random Search', 'Optuna (Bayesian)'],
    'Accuracy': [baseline_acc, grid_acc, random_acc, optuna_acc],
    'F1 Score': [baseline_f1, grid_f1, random_f1, optuna_f1],
    'CV Score': ['-', f"{grid_search.best_score_:.4f}", 
                 f"{random_search.best_score_:.4f}", f"{study.best_value:.4f}"]
})

print("="*60)
print("📊 HYPERPARAMETER TUNING COMPARISON")
print("="*60)
comparison

In [ ]:
# Best method
best_idx = comparison['F1 Score'].idxmax()
print(f"\n🏆 Best Method: {comparison.loc[best_idx, 'Method']}")
print(f"   F1 Score: {comparison.loc[best_idx, 'F1 Score']:.4f}")

## 7. Optuna Visualization

In [ ]:
# View trial history
trial_df = study.trials_dataframe()
print("📈 Top 10 Trials:")
trial_df.sort_values('value', ascending=False).head(10)[['number', 'value', 'params_n_estimators', 'params_max_depth']]

In [ ]:
# Parameter importance
try:
    importance = optuna.importance.get_param_importances(study)
    print("\n🔍 Parameter Importance:")
    for param, score in importance.items():
        print(f"   {param}: {score:.4f}")
except:
    print("Parameter importance requires more trials")

## ✅ Summary

This module provides:

**1. Grid Search**
- Exhaustive search over parameter grid
- Good for small search spaces
- Uses sklearn's `GridSearchCV`

**2. Random Search**
- Random sampling from parameter distributions
- More efficient for large search spaces
- Uses sklearn's `RandomizedSearchCV`

**3. Optuna (Bayesian Optimization)**
- Intelligent search using Tree-structured Parzen Estimator (TPE)
- Most efficient for complex search spaces
- Provides parameter importance analysis

**Key Takeaways:**
- Grid Search: Complete but slow
- Random Search: Faster, good results
- Optuna: Smart sampling, often best results